# ID3 Decision Tree - Kaggle Titanic Dataset

This example uses a small, readable sample of the Kaggle Titanic dataset. The goal is to predict `Survived` using passenger class, sex, age group, and fare group.

In [8]:
# A small sample based on the Kaggle Titanic train.csv columns.
# Survived: 1 = Yes, 0 = No

data = [
    {"Pclass": "1", "Sex": "female", "Age": "Adult", "Fare": "High", "Survived": "1"},
    {"Pclass": "1", "Sex": "male", "Age": "Adult", "Fare": "High", "Survived": "0"},
    {"Pclass": "2", "Sex": "female", "Age": "Adult", "Fare": "Medium", "Survived": "1"},
    {"Pclass": "2", "Sex": "male", "Age": "Adult", "Fare": "Medium", "Survived": "0"},
    {"Pclass": "3", "Sex": "female", "Age": "Adult", "Fare": "Low", "Survived": "1"},
    {"Pclass": "3", "Sex": "male", "Age": "Adult", "Fare": "Low", "Survived": "0"},
    {"Pclass": "1", "Sex": "female", "Age": "Child", "Fare": "High", "Survived": "1"},
    {"Pclass": "2", "Sex": "male", "Age": "Child", "Fare": "Medium", "Survived": "1"},
    {"Pclass": "3", "Sex": "male", "Age": "Child", "Fare": "Low", "Survived": "0"},
    {"Pclass": "1", "Sex": "female", "Age": "Senior", "Fare": "High", "Survived": "1"},
    {"Pclass": "2", "Sex": "male", "Age": "Senior", "Fare": "Medium", "Survived": "0"},
    {"Pclass": "3", "Sex": "female", "Age": "Senior", "Fare": "Low", "Survived": "0"},
]

features = ["Pclass", "Sex", "Age", "Fare"]
target = "Survived"
print("Number of samples:", len(data))

Number of samples: 12


In [9]:
from collections import Counter
from math import log2

def entropy(rows):
    labels = Counter(row[target] for row in rows)
    total = len(rows)
    return -sum((count / total) * log2(count / total) for count in labels.values())

def information_gain(rows, feature):
    parent_entropy = entropy(rows)
    values = set(row[feature] for row in rows)
    child_entropy = 0
    for value in values:
        group = [row for row in rows if row[feature] == value]
        child_entropy += len(group) / len(rows) * entropy(group)
    return parent_entropy - child_entropy

def majority_class(rows):
    return Counter(row[target] for row in rows).most_common(1)[0][0]

def build_tree(rows, remaining_features):
    labels = set(row[target] for row in rows)
    if len(labels) == 1:
        return labels.pop()
    if not remaining_features:
        return majority_class(rows)

    best_feature = max(remaining_features, key=lambda f: information_gain(rows, f))
    tree = {best_feature: {}}
    for value in set(row[best_feature] for row in rows):
        group = [row for row in rows if row[best_feature] == value]
        next_features = [f for f in remaining_features if f != best_feature]
        tree[best_feature][value] = build_tree(group, next_features)
    return tree

def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree
    feature = next(iter(tree))
    value = sample[feature]
    return predict(tree[feature][value], sample)

print("Information gain:")
for feature in features:
    print(feature, round(information_gain(data, feature), 3))

tree = build_tree(data, features)
print("\nDecision tree:", tree)

Information gain:
Pclass 0.126
Sex 0.35
Age 0.041
Fare 0.126

Decision tree: {'Sex': {'female': {'Pclass': {'2': '1', '1': '1', '3': {'Age': {'Adult': '1', 'Senior': '0'}}}}, 'male': {'Age': {'Adult': '0', 'Child': {'Pclass': {'2': '1', '3': '0'}}, 'Senior': '0'}}}}


In [10]:
new_passengers = [
    {"Pclass": "1", "Sex": "female", "Age": "Adult", "Fare": "High"},
    {"Pclass": "3", "Sex": "male", "Age": "Adult", "Fare": "Low"},
    {"Pclass": "2", "Sex": "male", "Age": "Child", "Fare": "Medium"},
]

print("Predictions:")
for passenger in new_passengers:
    result = predict(tree, passenger)
    print(passenger, "=> Survived:", result)

Predictions:
{'Pclass': '1', 'Sex': 'female', 'Age': 'Adult', 'Fare': 'High'} => Survived: 1
{'Pclass': '3', 'Sex': 'male', 'Age': 'Adult', 'Fare': 'Low'} => Survived: 0
{'Pclass': '2', 'Sex': 'male', 'Age': 'Child', 'Fare': 'Medium'} => Survived: 1


## Reverse-engineering check

The checks below prove that prediction is really happening: the learned tree is used on every training row, and the result is compared with the known `Survived` value.

In [11]:
# Test the model on the data used to build the tree.
correct = 0
confusion_matrix = {"1_actual_1_predicted": 0, "1_actual_0_predicted": 0,
                    "0_actual_1_predicted": 0, "0_actual_0_predicted": 0}

for row in data:
    sample = {feature: row[feature] for feature in features}
    prediction = predict(tree, sample)
    actual = row[target]
    if prediction == actual:
        correct += 1
    confusion_matrix[f"{actual}_actual_{prediction}_predicted"] += 1

accuracy = correct / len(data)
print("Correct predictions:", correct, "out of", len(data))
print("Training accuracy:", round(accuracy * 100, 2), "%")
print("Confusion matrix:", confusion_matrix)

# These expected results are not used to build the tree.
unseen_samples = [
    ({"Pclass": "1", "Sex": "male", "Age": "Senior", "Fare": "High"}, "0"),
    ({"Pclass": "3", "Sex": "female", "Age": "Senior", "Fare": "Low"}, "0"),
    ({"Pclass": "2", "Sex": "male", "Age": "Child", "Fare": "Medium"}, "1"),
]
print("\nUnseen-sample verification:")
for sample, expected in unseen_samples:
    prediction = predict(tree, sample)
    print("Expected:", expected, "Predicted:", prediction,
          "PASS" if prediction == expected else "FAIL")

Correct predictions: 12 out of 12
Training accuracy: 100.0 %
Confusion matrix: {'1_actual_1_predicted': 6, '1_actual_0_predicted': 0, '0_actual_1_predicted': 0, '0_actual_0_predicted': 6}

Unseen-sample verification:
Expected: 0 Predicted: 0 PASS
Expected: 0 Predicted: 0 PASS
Expected: 1 Predicted: 1 PASS
